# Jailbreak-type labelling models

Five one-vs-rest LightGBM classifiers, one per evaluated jailbreak type.
Each model is scored on the test fold that belongs to that type. Accuracy, F1,
AUC, TPR, and FNR are all saved. Ethical Appeal has 10 positives in the
168-row hand-labelled file, so accuracy alone is not a useful summary.

Set `USE_BERT_FEATURES=0` to train on TF-IDF instead of BERT CLS embeddings.
The TF-IDF vectoriser is fit on the training fold only. Auto-labelling of the
unlabelled DAN prompts is controlled by `LABEL_UNLABELLED` (default on).

In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "code").is_dir() else HERE.parent
sys.path.insert(0, str(ROOT / "code"))

import numpy as np
import pandas as pd
import torch
from lightgbm import LGBMClassifier
from tqdm import tqdm
from transformers import BertModel, BertTokenizer

from category_models import (
    fit_and_evaluate_precomputed,
    fit_and_evaluate_texts,
    one_vs_rest_frame,
)
from labels import EVALUATED_JAILBREAK_TYPES, parse_labels
from latex_tables import export_results
from paths import data_path, results_path

USE_BERT_FEATURES = os.environ.get("USE_BERT_FEATURES", "1") == "1"
LABEL_UNLABELLED = os.environ.get("LABEL_UNLABELLED", "1") == "1"

labelled = pd.read_csv(data_path("df_jailbreak_categ.csv"))
unlabelled_source = pd.read_csv(data_path("jailbreak_prompts_DAN.csv"))
frames = {name: one_vs_rest_frame(labelled, name) for name in EVALUATED_JAILBREAK_TYPES}
for name, frame in frames.items():
    print(f"{name}: positives={int(frame['Jailbreak'].sum())} of {len(frame)}")

def lgbm_factory():
    return LGBMClassifier(n_estimators=100, random_state=42, n_jobs=1, verbosity=-1)

rows = []
fitted = {}
if USE_BERT_FEATURES:
    base_prompts = next(iter(frames.values()))["prompt"].tolist()
    for frame in frames.values():
        if frame["prompt"].tolist() != base_prompts:
            raise RuntimeError("Category frames are not prompt-aligned; refuse to share embeddings.")
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    bert_model = BertModel.from_pretrained("bert-base-uncased")
    bert_model.eval()

    def extract_bert_features(texts, tokenizer, model, max_length=128):
        embeddings = []
        for text in tqdm(texts, desc="Extracting BERT features"):
            tokens = tokenizer.encode_plus(
                str(text),
                max_length=max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            with torch.no_grad():
                output = model(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"])
            embeddings.append(output.last_hidden_state[:, 0, :].squeeze().numpy())
        return np.vstack(embeddings)

    features = extract_bert_features(base_prompts, tokenizer, bert_model)
    for name, frame in frames.items():
        model, row = fit_and_evaluate_precomputed(
            name, features, frame["Jailbreak"], lgbm_factory, random_state=42
        )
        fitted[name] = {"model": model, "vectorizer": None, "features_for_unlabelled": "bert"}
        rows.append(row)
        print(name, {key: row[key] for key in ("accuracy", "f1", "auc", "tpr", "fnr")})
    unlabelled_features = None
else:
    features = None
    tokenizer = None
    bert_model = None
    for name, frame in frames.items():
        model, vectorizer, row = fit_and_evaluate_texts(
            name, frame["prompt"], frame["Jailbreak"], lgbm_factory, random_state=42
        )
        fitted[name] = {"model": model, "vectorizer": vectorizer}
        rows.append(row)
        print(name, {key: row[key] for key in ("accuracy", "f1", "auc", "tpr", "fnr")})

stem = "06_label_all_bert_lgbm" if USE_BERT_FEATURES else "06_label_all_tfidf_lgbm"
paths = export_results(stem, "category", rows)
print(f"Wrote {paths['tex']}")

if LABEL_UNLABELLED:
    pending = unlabelled_source[unlabelled_source["Labels"].map(lambda value: len(parse_labels(value)) == 0)].copy()
    if USE_BERT_FEATURES:
        pending_features = extract_bert_features(pending["prompt"], tokenizer, bert_model)
    predicted = []
    for row_index in range(len(pending)):
        names = []
        for name, bundle in fitted.items():
            if USE_BERT_FEATURES:
                features_row = pending_features[row_index : row_index + 1]
            else:
                features_row = bundle["vectorizer"].transform([pending["prompt"].iloc[row_index]])
            if int(bundle["model"].predict(features_row)[0]) == 1:
                names.append(name)
        predicted.append(", ".join(names))
    pending = pending.copy()
    pending["Labels"] = predicted
    output_path = results_path("auto_labelled_prompts.csv")
    pending.to_csv(output_path, index=False)
    print(f"Wrote {len(pending)} auto-labelled prompts to {output_path}")